# ACE ML Models - Model Registry

This notebook trains ML models for the ACE Intelligence Agent:
- **Service Request Volume Forecasting** - Predict future monthly service request volume
- **Member Churn Prediction** - Classify members at risk of non-renewal or cancellation
- **Response Success Prediction** - Predict service fulfillment success based on conditions

All models are registered to Snowflake Model Registry and can be added as tools to the Intelligence Agent.

## Prerequisites

**Required Packages** (configured automatically):
- `snowflake-ml-python`
- `scikit-learn`
- `xgboost`
- `matplotlib`

**Database Context:**
- **Database:** AAA_INTELLIGENCE  
- **Schema:** ANALYTICS  
- **Warehouse:** AAA_WH

**Note:** This notebook uses Snowflake Model Registry. Ensure you have appropriate permissions to create and register models.


## Import Required Packages


In [ ]:
# Import Python packages
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Import Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.snowpark import Window

# Import Snowpark ML
from snowflake.ml.modeling.preprocessing import StandardScaler, OneHotEncoder
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.linear_model import LinearRegression, LogisticRegression
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.modeling.metrics import mean_squared_error, mean_absolute_error, accuracy_score, roc_auc_score
from snowflake.ml.registry import Registry

print("✅ Packages imported successfully")


## Connect to Snowflake

Get active session and set context to ACE database.


In [ ]:
# Get active Snowflake session
session = get_active_session()

# Set context
session.use_database('AAA_INTELLIGENCE')
session.use_schema('ANALYTICS')
session.use_warehouse('AAA_WH')

print(f"✅ Connected - Role: {session.get_current_role()}")
print(f"   Warehouse: {session.get_current_warehouse()}")
print(f"   Database.Schema: {session.get_fully_qualified_current_schema()}")


---
# MODEL 1: Service Request Volume Forecasting

Predict future monthly service request volume using historical order data.


### Prepare Revenue Training Data


In [ ]:
# Get monthly service request volume data with features
service_volume_df = session.sql("""
SELECT
    DATE_TRUNC('month', request_timestamp)::DATE AS service_month,
    MONTH(request_timestamp) AS month_num,
    YEAR(request_timestamp) AS year_num,
    COUNT(DISTINCT service_id)::FLOAT AS total_service_requests,
    COUNT(DISTINCT member_id)::FLOAT AS unique_members,
    COUNT(DISTINCT vehicle_id)::FLOAT AS unique_vehicles,
    AVG(CASE WHEN priority = 'HIGH' THEN 1.0 ELSE 0.0 END)::FLOAT AS high_priority_ratio,
    COUNT(DISTINCT CASE WHEN service_type = 'TOWING' THEN service_id END)::FLOAT AS towing_count,
    COUNT(DISTINCT CASE WHEN service_type = 'TIRE_CHANGE' THEN service_id END)::FLOAT AS tire_count,
    COUNT(DISTINCT CASE WHEN service_type = 'BATTERY_JUMP' THEN service_id END)::FLOAT AS battery_count,
    AVG(CASE WHEN weather_condition IN ('RAIN', 'SNOW', 'ICE') THEN 1.0 ELSE 0.0 END)::FLOAT AS bad_weather_ratio
FROM RAW.SERVICE_REQUESTS
WHERE request_timestamp >= DATEADD('month', -36, CURRENT_DATE())
  AND request_timestamp < CURRENT_DATE()
GROUP BY DATE_TRUNC('month', request_timestamp), MONTH(request_timestamp), YEAR(request_timestamp)
ORDER BY service_month
""")

print(f"Service volume data: {service_volume_df.count()} months")
service_volume_df.show(5)

### Split Data and Train Revenue Model


In [ ]:
# Get monthly service request volume data with features
service_volume_df = session.sql("""
SELECT
    DATE_TRUNC('month', request_timestamp)::DATE AS service_month,
    MONTH(request_timestamp) AS month_num,
    YEAR(request_timestamp) AS year_num,
    COUNT(DISTINCT service_id)::FLOAT AS total_service_requests,
    COUNT(DISTINCT member_id)::FLOAT AS unique_members,
    COUNT(DISTINCT vehicle_id)::FLOAT AS unique_vehicles,
    AVG(CASE WHEN priority = 'HIGH' THEN 1.0 ELSE 0.0 END)::FLOAT AS high_priority_ratio,
    COUNT(DISTINCT CASE WHEN service_type = 'TOWING' THEN service_id END)::FLOAT AS towing_count,
    COUNT(DISTINCT CASE WHEN service_type = 'TIRE_CHANGE' THEN service_id END)::FLOAT AS tire_count,
    COUNT(DISTINCT CASE WHEN service_type = 'BATTERY_JUMP' THEN service_id END)::FLOAT AS battery_count,
    AVG(CASE WHEN weather_condition IN ('RAIN', 'SNOW', 'ICE') THEN 1.0 ELSE 0.0 END)::FLOAT AS bad_weather_ratio
FROM RAW.SERVICE_REQUESTS
WHERE request_timestamp >= DATEADD('month', -36, CURRENT_DATE())
  AND request_timestamp < CURRENT_DATE()
GROUP BY DATE_TRUNC('month', request_timestamp), MONTH(request_timestamp), YEAR(request_timestamp)
ORDER BY service_month
""")

print(f"Service volume data: {service_volume_df.count()} months")
service_volume_df.show(5)

### Evaluate and Register Revenue Model


In [ ]:
# Make predictions on test set
test_predictions = evidence_upload_volume_pipeline.predict(test_evidence_upload_volume)

# Calculate metrics
mae = mean_absolute_error(df=test_predictions, y_true_col_names="TOTAL_EVIDENCE_UPLOAD_VOLUME", y_pred_col_names="PREDICTED_EVIDENCE_UPLOAD_VOLUME")
mse = mean_squared_error(df=test_predictions, y_true_col_names="TOTAL_EVIDENCE_UPLOAD_VOLUME", y_pred_col_names="PREDICTED_EVIDENCE_UPLOAD_VOLUME")
rmse = mse ** 0.5

metrics = {"mae": round(mae, 2), "rmse": round(rmse, 2)}
print(f"Model metrics: {metrics}")

# Register model
reg = Registry(session)
reg.log_model(
    model=evidence_upload_volume_pipeline,
    model_name="SERVICE_VOLUME_PREDICTOR",
    version_name="V1",
    comment="Predicts monthly evidence upload volume based on historical deployment and storage patterns using Linear Regression",
    metrics=metrics
)

print("✅ Evidence volume model registered to Model Registry as SERVICE_VOLUME_PREDICTOR")


---
# MODEL 2: Member Churn Prediction

Classify agencies as likely to churn or not based on behavior patterns.


### Prepare Churn Training Data


In [ ]:
# Get member features for churn prediction
churn_df = session.sql("""
SELECT
    m.member_id,
    m.membership_level,
    m.risk_score::FLOAT AS risk_score,
    m.lifetime_value::FLOAT AS lifetime_value,
    DATEDIFF('day', m.membership_start_date, CURRENT_DATE())::FLOAT AS membership_days,
    DATEDIFF('day', CURRENT_DATE(), m.membership_renewal_date)::FLOAT AS days_to_renewal,
    m.is_auto_renew::BOOLEAN AS is_auto_renew,
    -- Service usage patterns (last 6 months)
    COUNT(DISTINCT sr.service_id)::FLOAT AS service_requests_6m,
    COUNT(DISTINCT CASE WHEN sr.service_type = 'TOWING' THEN sr.service_id END)::FLOAT AS towing_requests_6m,
    -- Service fulfillment metrics
    AVG(sf.response_time_minutes)::FLOAT AS avg_response_time,
    AVG(sf.member_satisfaction_score)::FLOAT AS avg_satisfaction_score,
    COUNT(DISTINCT CASE WHEN sf.member_satisfaction_score <= 2 THEN sf.service_id END)::FLOAT AS low_satisfaction_count,
    -- Transaction history
    COUNT(DISTINCT mt.transaction_id)::FLOAT AS total_transactions,
    SUM(CASE WHEN mt.transaction_type = 'RENEWAL' THEN 1 ELSE 0 END)::FLOAT AS renewal_count,
    -- Predictive scores
    AVG(ps.churn_risk_score)::FLOAT AS avg_churn_risk_score,
    -- Target: Is churned (membership status)
    (m.membership_status = 'CANCELLED')::BOOLEAN AS is_churned
FROM RAW.MEMBERS m
LEFT JOIN RAW.SERVICE_REQUESTS sr ON m.member_id = sr.member_id 
    AND sr.request_timestamp >= DATEADD('month', -6, CURRENT_DATE())
LEFT JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id
LEFT JOIN RAW.MEMBER_TRANSACTIONS mt ON m.member_id = mt.member_id
LEFT JOIN RAW.PREDICTIVE_SCORES ps ON m.member_id = ps.member_id
WHERE m.membership_status IN ('ACTIVE', 'CANCELLED')
  AND m.membership_start_date <= DATEADD('month', -12, CURRENT_DATE()) -- At least 1 year old
GROUP BY m.member_id, m.membership_level, m.risk_score, m.lifetime_value, 
         m.membership_start_date, m.membership_renewal_date, m.is_auto_renew, m.membership_status
HAVING COUNT(DISTINCT sr.service_id) > 0 OR COUNT(DISTINCT mt.transaction_id) > 0
LIMIT 10000  -- Limit for faster training
""")

print(f"Churn data: {churn_df.count()} members")
churn_df.show(5)

### Train Churn Classification Model


In [ ]:
# Train/test split (80/20)
train_churn, test_churn = churn_df.random_split([0.8, 0.2], seed=42)

# Drop AGENCY_ID
train_churn = train_churn.drop("AGENCY_ID")
test_churn = test_churn.drop("AGENCY_ID")

# Create pipeline with preprocessing and classification
churn_pipeline = Pipeline([
    ("Encoder", OneHotEncoder(
        input_cols=["AGENCY_SEGMENT", "JURISDICTION"],
        output_cols=["AGENCY_SEGMENT_ENCODED", "JURISDICTION_ENCODED"],
        drop_input_cols=True,  # Drop original string columns after encoding
        handle_unknown="ignore"
    )),
    ("Classifier", RandomForestClassifier(
        label_cols=["IS_CHURNED"],
        output_cols=["CHURN_PREDICTION"],
        n_estimators=100,
        max_depth=10
    ))
])

# Train model
churn_pipeline.fit(train_churn)
print("✅ Churn classification model trained")


### Evaluate and Register Churn Model


In [ ]:
# Make predictions
churn_predictions = churn_pipeline.predict(test_churn)

# Calculate metrics
accuracy = accuracy_score(df=churn_predictions, y_true_col_names="IS_CHURNED", y_pred_col_names="CHURN_PREDICTION")
# Note: ROC AUC might need probability scores - using accuracy for now
churn_metrics = {"accuracy": round(accuracy, 4)}
print(f"Churn model metrics: {churn_metrics}")

# Register model (use different name to avoid conflict)
reg.log_model(
    model=churn_pipeline,
    model_name="MEMBER_CHURN_PREDICTOR",
    version_name="V1",
    comment="Predicts member churn probability using Random Forest based on behavior patterns",
    metrics=churn_metrics
)

print("✅ Churn model registered to Model Registry as MEMBER_CHURN_PREDICTOR")


---
# MODEL 3: Response Success Prediction Prediction

Predict which design wins are likely to convert to production orders.


### Prepare Response Success Prediction Data


In [ ]:
# Get service request features for success prediction
response_success_df = session.sql("""
SELECT
    sr.service_id,
    sr.service_type,
    sr.service_category,
    sr.priority,
    sr.location_type,
    sr.weather_condition,
    sr.temperature_f::FLOAT AS temperature_f,
    sr.channel,
    -- Time features
    HOUR(sr.request_timestamp)::INT AS request_hour,
    DAYOFWEEK(sr.request_timestamp)::INT AS request_dow,
    -- Regional features
    reg.average_response_time_minutes::FLOAT AS region_avg_response,
    reg.active_technicians::FLOAT AS region_technicians,
    reg.active_trucks::FLOAT AS region_trucks,
    -- Member features
    m.membership_level,
    v.vehicle_type,
    -- Technician features
    t.certification_level,
    t.average_response_time_minutes::FLOAT AS tech_avg_response,
    -- Success criteria: Completed within regional SLA and high satisfaction
    (sf.service_outcome = 'COMPLETED' 
     AND sf.response_time_minutes <= reg.target_response_time_minutes
     AND (sf.member_satisfaction_score >= 4 OR sf.member_satisfaction_score IS NULL))::BOOLEAN AS response_successful
FROM RAW.SERVICE_REQUESTS sr
JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id
JOIN RAW.SERVICE_TECHNICIANS t ON sf.technician_id = t.technician_id
JOIN RAW.MEMBERS m ON sr.member_id = m.member_id
LEFT JOIN RAW.VEHICLES v ON sr.vehicle_id = v.vehicle_id
LEFT JOIN RAW.SERVICE_REGIONS reg ON t.service_region = reg.region_name
WHERE sr.request_timestamp >= DATEADD('month', -12, CURRENT_DATE())
  AND sf.completion_timestamp IS NOT NULL
""")

print(f"Response success data: {response_success_df.count()} service requests")
response_success_df.show(5)

### Train Conversion Model


In [ ]:
# Get service request features for success prediction
response_success_df = session.sql("""
SELECT
    sr.service_id,
    sr.service_type,
    sr.service_category,
    sr.priority,
    sr.location_type,
    sr.weather_condition,
    sr.temperature_f::FLOAT AS temperature_f,
    sr.channel,
    -- Time features
    HOUR(sr.request_timestamp)::INT AS request_hour,
    DAYOFWEEK(sr.request_timestamp)::INT AS request_dow,
    -- Regional features
    reg.average_response_time_minutes::FLOAT AS region_avg_response,
    reg.active_technicians::FLOAT AS region_technicians,
    reg.active_trucks::FLOAT AS region_trucks,
    -- Member features
    m.membership_level,
    v.vehicle_type,
    -- Technician features
    t.certification_level,
    t.average_response_time_minutes::FLOAT AS tech_avg_response,
    -- Success criteria: Completed within regional SLA and high satisfaction
    (sf.service_outcome = 'COMPLETED' 
     AND sf.response_time_minutes <= reg.target_response_time_minutes
     AND (sf.member_satisfaction_score >= 4 OR sf.member_satisfaction_score IS NULL))::BOOLEAN AS response_successful
FROM RAW.SERVICE_REQUESTS sr
JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id
JOIN RAW.SERVICE_TECHNICIANS t ON sf.technician_id = t.technician_id
JOIN RAW.MEMBERS m ON sr.member_id = m.member_id
LEFT JOIN RAW.VEHICLES v ON sr.vehicle_id = v.vehicle_id
LEFT JOIN RAW.SERVICE_REGIONS reg ON t.service_region = reg.region_name
WHERE sr.request_timestamp >= DATEADD('month', -12, CURRENT_DATE())
  AND sf.completion_timestamp IS NOT NULL
""")

print(f"Response success data: {response_success_df.count()} service requests")
response_success_df.show(5)

### Evaluate and Register Conversion Model


In [ ]:
# Predict on test set
deployment_success_predictions = deployment_success_pipeline.predict(test_deployment_success)

# Calculate accuracy
success_accuracy = accuracy_score(df=deployment_success_predictions, 
                                   y_true_col_names="DEPLOYMENT_SUCCESSFUL",
                                   y_pred_col_names="SUCCESS_PREDICTION")
success_metrics = {"accuracy": round(success_accuracy, 4)}
print(f"Deployment success model metrics: {success_metrics}")

# Register model
reg.log_model(
    model=deployment_success_pipeline,
    model_name="DEPLOYMENT_SUCCESS_PREDICTOR",
    version_name="V1",
    comment="Predicts deployment success (active with evidence uploads) using Logistic Regression based on officer, agency, and product features",
    metrics=success_metrics
)

print("✅ Deployment success model registered to Model Registry as DEPLOYMENT_SUCCESS_PREDICTOR")


---
# Verify Models in Registry


In [ ]:
# Show all models in the registry
print("Models in registry:")
reg.show_models()

# Show versions for evidence volume model
print("\nService Volume Predictor versions:")
reg.get_model("SERVICE_VOLUME_PREDICTOR").show_versions()

# Show versions for churn model  
print("\nMember Churn Predictor versions:")
reg.get_model("MEMBER_CHURN_PREDICTOR").show_versions()

# Show versions for deployment success model
print("\nDeployment Success Predictor versions:")
reg.get_model("DEPLOYMENT_SUCCESS_PREDICTOR").show_versions()

print("\n✅ All models registered and ready to add to Intelligence Agent")


---
# Test Model Inference

Test calling each model to make predictions.


In [ ]:
# Get monthly service request volume data with features
service_volume_df = session.sql("""
SELECT
    DATE_TRUNC('month', request_timestamp)::DATE AS service_month,
    MONTH(request_timestamp) AS month_num,
    YEAR(request_timestamp) AS year_num,
    COUNT(DISTINCT service_id)::FLOAT AS total_service_requests,
    COUNT(DISTINCT member_id)::FLOAT AS unique_members,
    COUNT(DISTINCT vehicle_id)::FLOAT AS unique_vehicles,
    AVG(CASE WHEN priority = 'HIGH' THEN 1.0 ELSE 0.0 END)::FLOAT AS high_priority_ratio,
    COUNT(DISTINCT CASE WHEN service_type = 'TOWING' THEN service_id END)::FLOAT AS towing_count,
    COUNT(DISTINCT CASE WHEN service_type = 'TIRE_CHANGE' THEN service_id END)::FLOAT AS tire_count,
    COUNT(DISTINCT CASE WHEN service_type = 'BATTERY_JUMP' THEN service_id END)::FLOAT AS battery_count,
    AVG(CASE WHEN weather_condition IN ('RAIN', 'SNOW', 'ICE') THEN 1.0 ELSE 0.0 END)::FLOAT AS bad_weather_ratio
FROM RAW.SERVICE_REQUESTS
WHERE request_timestamp >= DATEADD('month', -36, CURRENT_DATE())
  AND request_timestamp < CURRENT_DATE()
GROUP BY DATE_TRUNC('month', request_timestamp), MONTH(request_timestamp), YEAR(request_timestamp)
ORDER BY service_month
""")

print(f"Service volume data: {service_volume_df.count()} months")
service_volume_df.show(5)

---
# Next Steps

## Add Models to Intelligence Agent

**Option 1: Using the SQL Script (Easiest)**
Run `sql/agent/08_create_intelligence_agent.sql` which automatically configures all 3 ML models.

**Option 2: Manual Configuration in Snowsight**
1. In Snowsight → AI & ML → Agents → AAA_INTELLIGENCE_AGENT
2. Go to Tools → + Add → Function
3. Add each model wrapper procedure:
   - **PREDICT_EVIDENCE_UPLOAD_VOLUME** (from `sql/ml/07_create_model_wrapper_functions.sql`)
   - **PREDICT_AGENCY_CHURN** (from `sql/ml/07_create_model_wrapper_functions.sql`)
   - **PREDICT_DEPLOYMENT_SUCCESS** (from `sql/ml/07_create_model_wrapper_functions.sql`)

## Example Questions for Agent

- "Predict evidence upload volume for the next 6 months"
- "Which agencies are at high risk of churn?"
- "What is the predicted success rate for deploying Body Camera 3 to Officer OFC00012345?"
- "Forecast storage needs for Evidence.com over the next quarter"

The models will now be available as tools your agent can use!
